# PSPNet (ADE20K) guidance caches for CDnet2014, LIMU and WallFlower: inference only, no training

Same recipe as the DeepLabV3+ caches. **Settings:** Accelerator **GPU T4** (not P100), **Internet ON**, *Add Input*: your CDnet2014, LIMU and WallFlower datasets.

For the long CDnet run use **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed (limit 12 h). Outputs appear in the version's *Output* tab.

CDnet frames match the DeepLab cache exactly: every **2nd** temporal-ROI frame, longest side **320 px**, PTZ skipped. LIMU: every frame from 400. WallFlower: Train+Run+Evaluate window.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo CDnet temporalROI files: $(find /kaggle/input -name temporalROI.txt | wc -l)
!find /kaggle/input -maxdepth 6 -type d \( -name CameraParameter -o -name Intersection -o -name LightSwitch \) | head
!echo WallFlower script.txt files: $(find /kaggle/input -name script.txt | wc -l)

## 1. Environment (Python 3.10, torch 2.1, mmcv 2.1, mmsegmentation 1.2.2)

In [ ]:
%%bash
set -e
PREFIX=/tmp/mc; PY=$PREFIX/envs/mm/bin/python; PIP=$PREFIX/envs/mm/bin/pip
if [ ! -x $PY ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/mc.sh && bash /tmp/mc.sh -b -p $PREFIX >/dev/null
  $PREFIX/bin/conda create -y -q -n mm python=3.10 -c conda-forge --override-channels >/dev/null
fi
$PIP install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
$PIP install -q -U openmim && $PY -m mim install -q "mmengine" "mmcv==2.1.0"
$PIP install -q "mmsegmentation==1.2.2" ftfy regex
$PIP uninstall -y -q opencv-python opencv-python-headless numpy >/dev/null 2>&1 || true
$PIP install -q --no-deps "numpy==1.26.4" "opencv-python-headless==4.9.0.80"
$PY -c "import torch, mmcv, mmseg; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| mmcv', mmcv.__version__, '| mmseg', mmseg.__version__)"

## 2. PSPNet R50 ADE20K checkpoint (pre-trained, no training)

In [ ]:
%%bash
/tmp/mc/envs/mm/bin/python -m mim download mmsegmentation --config pspnet_r50-d8_4xb4-160k_ade20k-512x512 --dest /tmp/psp
ls -la /tmp/psp

## 3. The three guidance scripts

In [ ]:
%%writefile /kaggle/working/make_guidance_cdnet_mmseg.py
#!/usr/bin/env python3
"""
DENSE ADE20K guidance caches for CDnet2014 (GPU), using YOUR corrected setup:
  * the paper's 12 foreground classes (fixed ADE20K ids),
  * BGR frame passed straight to inference_model (mmseg converts internally),
  * DeepLabV3+ (your working model) or PSPNet,
  * guidance at the frame's NATIVE resolution.
Writes, per scene, <out_dir>/<category>__<scene>.npz with:
  nums uint32[N], guid uint8[N,H,W] (0/255), prob float16[N,H,W] in [0,1]
— the exact format the CPB framework reads. Runs over the FULL temporalROI range.
"""
import os, re, argparse
import numpy as np, cv2, torch
from mmseg.apis import init_model, inference_model
from mmseg.utils import register_all_modules

# The paper's 12 foreground classes, canonical 0-indexed MMSeg ADE20K ids
ADE20K_FG_CLASSES = {12:'person',20:'car',39:'cushion',41:'box',67:'book',76:'boat',
                     80:'bus',83:'truck',98:'bottle',102:'van',115:'bag',127:'bicycle'}
FG_IDS = np.array(sorted(ADE20K_FG_CLASSES.keys()), dtype=np.int64)

def list_nums(d):
    ns = []
    for fn in os.listdir(d):
        if fn.lower().endswith(('.jpg', '.png')):
            m = re.findall(r'\d+', fn)
            if m: ns.append(int(m[-1]))
    return sorted(ns)

def temporal_roi(sp):
    a, b = open(os.path.join(sp, 'temporalROI.txt')).read().split()[:2]; return int(a), int(b)

def load_frame(idir, n):
    for pat in (f'in{n:06d}.jpg', f'in{n:05d}.jpg', f'{n:06d}.jpg', f'in{n:06d}.png'):
        p = os.path.join(idir, pat)
        if os.path.exists(p):
            im = cv2.imread(p)
            if im is not None: return im
    return None

def predict(model, frame_bgr):
    H, W = frame_bgr.shape[:2]
    res = inference_model(model, frame_bgr)                 # BGR in (bgr_to_rgb handled internally)
    pred = res.pred_sem_seg.data.squeeze().cpu().numpy()
    if pred.ndim > 2: pred = pred[0]
    mask = np.isin(pred, FG_IDS).astype(np.uint8) * 255
    try:
        logits = res.seg_logits.data                        # [C,h,w]
        probs = torch.softmax(logits.float(), dim=0)
        fg = probs[torch.as_tensor(FG_IDS, device=probs.device)].amax(0).cpu().numpy().astype(np.float32)
    except Exception:
        fg = (mask > 0).astype(np.float32)                  # fallback: binary prob
    if mask.shape[:2] != (H, W):
        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
    if fg.shape[:2] != (H, W):
        fg = cv2.resize(fg, (W, H))
    return mask, fg

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--dataset_root', required=True)
    ap.add_argument('--config', required=True, help='path to the mmseg config .py')
    ap.add_argument('--ckpt', required=True, help='path to the .pth checkpoint')
    ap.add_argument('--out_dir', default='cdnet_guidance_pspnet')
    ap.add_argument('--categories', default=None)
    ap.add_argument('--scenes', default=None)
    ap.add_argument('--stride', type=int, default=1, help='1 = every temporalROI frame')
    ap.add_argument('--max_dim', type=int, default=0, help='0 = native (matches your caches); else cap longest side')
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    register_all_modules()
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'device={dev}  config={args.config}')
    model = init_model(args.config, args.ckpt, device=dev); model.eval()
    print('model loaded OK; FG ids =', FG_IDS.tolist())

    root = args.dataset_root
    if args.scenes:
        sids = [s.strip() for s in args.scenes.split(',')]
    else:
        cats = [c.strip() for c in args.categories.split(',')] if args.categories else \
               sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
        sids = [f'{c}/{s}' for c in cats
                for s in sorted(os.listdir(os.path.join(root, c)))
                if os.path.isdir(os.path.join(root, c, s, 'input'))]

    for sid in sids:
        cat, scene = sid.split('/')
        sp = os.path.join(root, cat, scene); idir = os.path.join(sp, 'input')
        out = os.path.join(args.out_dir, f'{cat}__{scene}.npz')
        if os.path.exists(out):
            print(f'[skip] {sid} exists'); continue
        a, b = temporal_roi(sp)
        nums = [n for n in list_nums(idir) if a <= n <= b][::args.stride]
        G, P, K = [], [], []
        with torch.no_grad():
            for i, n in enumerate(nums):
                im = load_frame(idir, n)
                if im is None: continue
                mask, fg = predict(model, im)
                if args.max_dim and args.max_dim > 0:
                    h, w = mask.shape; sc = args.max_dim / max(h, w)
                    if sc < 1.0:
                        nw, nh = int(round(w*sc)), int(round(h*sc))
                        mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
                        fg = cv2.resize(fg, (nw, nh))
                G.append(mask.astype(np.uint8)); P.append(fg.astype(np.float16)); K.append(n)
                if i % 200 == 0: print(f'  {sid} {i}/{len(nums)}')
        if not K:
            print(f'[warn] {sid} no frames'); continue
        np.savez_compressed(out, nums=np.array(K, np.int32), guid=np.stack(G), prob=np.stack(P))
        print(f'[saved] {out}  N={len(K)}  {os.path.getsize(out)/1e6:.0f} MB')

if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/make_guidance_limu.py
#!/usr/bin/env python3
"""
make_guidance_limu.py -- ADE20K semantic guidance caches for LIMU (GPU, Kaggle/Colab). Inference only, no training.
Same conventions as the CDnet / WallFlower caches: the paper's 12 ADE20K foreground classes, BGR frame passed to
mmseg inference_model, guidance at native resolution (LIMU = 320x240).
Frames: every frame from --start (default 400 = first frame after the CPB training window) to the end.
Output per sequence <out_dir>/<Sequence>.npz:
  nums int32[N] | guid uint8[N,H,W] 0/255 | prob float16[N,H,W] = max softmax over the 12 fg classes | meta (json)
Sequences are found by searching --limu_root for folders named CameraParameter, Intersection, LightSwitch.
Finished sequences are skipped (resume).
"""
import os, re, json, argparse, glob
import numpy as np, cv2, torch
from mmseg.apis import init_model, inference_model
from mmseg.utils import register_all_modules

ADE20K_FG = {12: 'person', 20: 'car', 39: 'cushion', 41: 'box', 67: 'book', 76: 'boat', 80: 'bus', 83: 'truck',
             98: 'bottle', 102: 'van', 115: 'bag', 127: 'bicycle'}
FG_IDS = np.array(sorted(ADE20K_FG), np.int64)
SEQS = ['CameraParameter', 'Intersection', 'LightSwitch']


def find_seq(root, name):
    for d, dirs, files in os.walk(root):
        if os.path.basename(d) == name and any(f.lower().endswith('.jpg') for f in files):
            return d
    return None


def predict(model, bgr):
    H, W = bgr.shape[:2]
    res = inference_model(model, bgr)
    pred = res.pred_sem_seg.data.squeeze().cpu().numpy()
    mask = np.isin(pred, FG_IDS).astype(np.uint8) * 255
    probs = torch.softmax(res.seg_logits.data.float(), dim=0)
    fg = probs[torch.as_tensor(FG_IDS, device=probs.device)].amax(0).cpu().numpy().astype(np.float32)
    if mask.shape != (H, W):
        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST); fg = cv2.resize(fg, (W, H))
    return mask, fg


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--limu_root', required=True); ap.add_argument('--config', required=True); ap.add_argument('--ckpt', required=True)
    ap.add_argument('--name', default='pspnet_r50_ade20k'); ap.add_argument('--out_dir', default='limu_guidance_pspnet')
    ap.add_argument('--start', type=int, default=400); ap.add_argument('--dry_run', action='store_true')
    a = ap.parse_args(); os.makedirs(a.out_dir, exist_ok=True)
    found = {s: find_seq(a.limu_root, s) for s in SEQS}
    print('sequences found:', found)
    if a.dry_run:
        return
    register_all_modules(); dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = init_model(a.config, a.ckpt, device=dev); model.eval(); print('model loaded on', dev)
    for s, d in found.items():
        out = os.path.join(a.out_dir, f'{s}.npz')
        if d is None:
            print(f'[warn] {s} not found'); continue
        if os.path.exists(out):
            print(f'[skip] {s} exists'); continue
        files = {}
        for p in glob.glob(os.path.join(d, '*.jpg')):
            files[int(re.findall(r'(\d+)', os.path.basename(p))[-1])] = p
        nums = [n for n in sorted(files) if n >= a.start]
        G, Pr, K = [], [], []
        with torch.no_grad():
            for i, n in enumerate(nums):
                m, f = predict(model, cv2.imread(files[n]))
                G.append(m); Pr.append(f.astype(np.float16)); K.append(n)
                if i % 500 == 0:
                    print(f'  {s} {i}/{len(nums)}', flush=True)
        meta = json.dumps(dict(sequence=s, model=a.name, config=os.path.basename(a.config), ckpt=os.path.basename(a.ckpt),
                               frames=f'{a.start}-end', fg_ids=FG_IDS.tolist(), fg_classes=ADE20K_FG, prob='float16 max fg-class softmax'))
        np.savez_compressed(out, nums=np.array(K, np.int32), guid=np.stack(G), prob=np.stack(Pr), meta=np.array(meta))
        print(f'[saved] {out}  N={len(K)}  {os.path.getsize(out) / 1e6:.0f} MB', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/make_guidance_wallflower.py
#!/usr/bin/env python3
"""
make_guidance_wallflower.py -- ADE20K semantic guidance caches for WallFlower (GPU, Kaggle/Colab)
================================================================================================
NO TRAINING. As in Liang et al. 2021, the segmentation network (DeepLabV3+ or PSPNet) is used
exactly as released, PRE-TRAINED on ADE20K, inference only. WallFlower has one ground-truth
frame per sequence, so fine-tuning on it would be both impossible and test-set leakage.

Same conventions as the CDnet caches you already used (make_guidance.py):
  * foreground = the paper's 12 ADE20K classes (fixed 0-indexed MMSeg ids, below);
  * BGR frame passed straight to mmseg's inference_model (it converts to RGB internally);
  * guidance stored at NATIVE resolution (WallFlower = 160x120).

Output, one file per sequence:  <out_dir>/<Sequence>.npz
    nums : int32   [N]        WallFlower frame numbers (b%05d.bmp)
    guid : uint8   [N,H,W]    0/255, argmax class is one of the 12 foreground classes
    prob : float16 [N,H,W]    max softmax probability over the 12 foreground classes, [0,1]
    meta : str                json: model, frames mode, fg ids, script.txt windows
plus  <out_dir>/check/<Sequence>_eval.png   (input | guidance | ground truth, for a visual check)
and   <out_dir>/guidance_summary.txt       (guidance-only F on the evaluated frame, per sequence)

--frames
  window (default) : from the first official Train frame to the Evaluate frame (train + run + eval)
                     -> 3,693 frames in total, enough for every experiment we plan on WallFlower
  run              : Run frames + Evaluate frame only (2,292 frames) -- the minimum
  all              : every frame of every sequence (16,158 frames) -- only if you want guidance
                     during the LightSwitch lights-on model (frames 0-811) as well

The script RESUMES: a sequence whose .npz already exists is skipped.
--dry_run replaces the network by an all-background predictor (tests I/O without a GPU).
"""
import os, re, json, time, argparse
import numpy as np, cv2

ADE20K_FG_CLASSES = {12: 'person', 20: 'car', 39: 'cushion', 41: 'box', 67: 'book',
                     76: 'boat', 80: 'bus', 83: 'truck', 98: 'bottle', 102: 'van',
                     115: 'bag', 127: 'bicycle'}
FG_IDS = np.array(sorted(ADE20K_FG_CLASSES.keys()), dtype=np.int64)
SEQS = ['Bootstrap', 'Camouflage', 'ForegroundAperture', 'LightSwitch', 'TimeOfDay', 'WavingTrees', 'MovedObject']


# ----------------------------------------------------------------------------- WallFlower layout
def find_sequences(root):
    """Walk `root` and return {sequence_name: folder} for every folder holding script.txt + b*.bmp."""
    found = {}
    for dp, dn, fn in os.walk(root, followlinks=True):
        if 'script.txt' in fn and any(f.lower().startswith('b') and f.lower().endswith('.bmp') for f in fn):
            found[os.path.basename(dp.rstrip('/'))] = dp
    return found


def frame_path(d, n):
    for pat in (f'b{n:05d}.bmp', f'b{n:05d}.BMP'):
        p = os.path.join(d, pat)
        if os.path.exists(p):
            return p
    return None


def all_frames(d):
    return sorted(int(m.group(1)) for f in os.listdir(d) for m in [re.match(r'b(\d{5})\.bmp$', f, re.I)] if m)


def script_windows(d):
    """Official protocol from script.txt -> (train_nums, run_nums, eval_idx). Same parser as wallflower_eval.py."""
    txt = open(os.path.join(d, 'script.txt')).read()
    tr = re.search(r'(\d+)\s*-\s*(\d+)\s+Train', txt)
    rn = re.search(r'(\d+)\s*-\s*(\d+)\s+Run', txt)
    ev = re.search(r'(\d+)\s+Evaluate', txt)
    a, b = int(tr.group(1)), int(tr.group(2))
    r0, r1 = (int(rn.group(1)), int(rn.group(2))) if rn else (b + 1, int(ev.group(1)))
    have = set(all_frames(d))
    train = [n for n in range(a, b + 1) if n in have]
    run = [n for n in range(max(r0, b + 1), r1) if n in have]
    return train, run, int(ev.group(1))


def load_gt(d):
    for f in os.listdir(d):
        if f.lower().startswith('hand_segmented'):
            return cv2.imread(os.path.join(d, f), cv2.IMREAD_GRAYSCALE)
    return None


# ----------------------------------------------------------------------------- model
class MMSegGuidance:
    def __init__(self, config, ckpt):
        import torch
        from mmseg.apis import init_model
        from mmseg.utils import register_all_modules
        register_all_modules()
        self.torch = torch
        self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = init_model(config, ckpt, device=self.dev)
        self.model.eval()
        self.fg = torch.as_tensor(FG_IDS, device=self.dev)

    def __call__(self, frames_bgr):
        from mmseg.apis import inference_model
        out = []
        with self.torch.no_grad():
            res = inference_model(self.model, frames_bgr)
        if not isinstance(res, (list, tuple)):
            res = [res]
        for im, r in zip(frames_bgr, res):
            H, W = im.shape[:2]
            pred = r.pred_sem_seg.data.squeeze().cpu().numpy()
            if pred.ndim > 2:
                pred = pred[0]
            mask = np.isin(pred, FG_IDS).astype(np.uint8) * 255
            try:
                probs = self.torch.softmax(r.seg_logits.data.float(), dim=0)
                prob = probs[self.fg].amax(0).cpu().numpy().astype(np.float32)
            except Exception:
                prob = (mask > 0).astype(np.float32)
            if mask.shape != (H, W):
                mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
            if prob.shape != (H, W):
                prob = cv2.resize(prob, (W, H), interpolation=cv2.INTER_LINEAR)
            out.append((mask, prob))
        return out


class DryRun:
    dev = 'none (dry run)'
    def __call__(self, frames_bgr):
        return [(np.zeros(f.shape[:2], np.uint8), np.zeros(f.shape[:2], np.float32)) for f in frames_bgr]


# ----------------------------------------------------------------------------- main
def f_measure(mask, gt):
    fg, g = mask > 127, gt > 127
    tp = int((fg & g).sum()); fp = int((fg & ~g).sum()); fn = int((~fg & g).sum()); tn = int((~fg & ~g).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    return f, p, r, spec


def main():
    ap = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    ap.add_argument('--wf_root', default='/kaggle/input', help='folder that CONTAINS the WallFlower sequences (searched recursively)')
    ap.add_argument('--config', default=None, help='mmseg config .py (from `mim download`)')
    ap.add_argument('--ckpt', default=None, help='mmseg checkpoint .pth')
    ap.add_argument('--name', default='deeplabv3plus', help='label stored in the cache metadata')
    ap.add_argument('--out_dir', default='wf_guidance_deeplabv3plus')
    ap.add_argument('--frames', choices=['window', 'run', 'all'], default='window')
    ap.add_argument('--seqs', default=','.join(SEQS))
    ap.add_argument('--batch', type=int, default=8)
    ap.add_argument('--dry_run', action='store_true')
    a = ap.parse_args()

    seqmap = find_sequences(a.wf_root)
    want = [s.strip() for s in a.seqs.split(',') if s.strip()]
    missing = [s for s in want if s not in seqmap]
    print(f'WallFlower sequences found under {a.wf_root}: {sorted(seqmap)}')
    if missing:
        raise SystemExit(f'Missing sequences: {missing}. Point --wf_root at the folder that contains them.')

    os.makedirs(os.path.join(a.out_dir, 'check'), exist_ok=True)
    net = DryRun() if a.dry_run else MMSegGuidance(a.config, a.ckpt)
    print(f'device = {net.dev}   model = {a.name}   frames = {a.frames}   foreground ids = {FG_IDS.tolist()}')

    summary = []
    t_all = time.time(); n_all = 0
    for seq in want:
        d = seqmap[seq]
        out = os.path.join(a.out_dir, f'{seq}.npz')
        tr, run, ev = script_windows(d)
        if a.frames == 'window':
            nums = [n for n in all_frames(d) if tr[0] <= n <= ev]
        elif a.frames == 'run':
            nums = run + [ev]
        else:
            nums = all_frames(d)
        if os.path.exists(out):
            print(f'[skip] {seq}: {out} already exists'); continue
        t0 = time.time()
        G, P = [], []
        for i in range(0, len(nums), a.batch):
            chunk = nums[i:i + a.batch]
            ims = [cv2.imread(frame_path(d, n)) for n in chunk]
            for m, p in net(ims):
                G.append(m); P.append(p.astype(np.float16))
            if (i // a.batch) % 25 == 0:
                print(f'  {seq}: {min(i + a.batch, len(nums))}/{len(nums)}  ({(min(i + a.batch, len(nums))) / max(time.time() - t0, 1e-6):.1f} frames/s)', flush=True)
        meta = dict(sequence=seq, model=a.name, config=a.config, ckpt=a.ckpt, frames=a.frames,
                    fg_ids=FG_IDS.tolist(), fg_classes=ADE20K_FG_CLASSES, train=[tr[0], tr[-1]],
                    run=[run[0], run[-1]] if run else None, eval=ev, native_hw=list(G[0].shape))
        np.savez_compressed(out, nums=np.array(nums, np.int32), guid=np.stack(G), prob=np.stack(P),
                            meta=np.array(json.dumps(meta)))
        n_all += len(nums)

        # sanity check on the evaluated frame (guidance ALONE, no CPB)
        gt = load_gt(d)
        k = nums.index(ev)
        f, p, r, spec = f_measure(G[k], gt)
        line = (f'{seq:20s} N={len(nums):5d}  {time.time() - t0:6.1f}s  eval frame {ev}: guidance-only '
                + (f'specificity={spec:.3f}' if seq == 'MovedObject' else f'F={f:.3f} P={p:.2f} R={r:.2f}')
                + f'  guidance fg {100 * np.mean(G[k] > 127):.1f}%  GT fg {100 * np.mean(gt > 127):.1f}%')
        print('[saved] ' + line, flush=True)
        summary.append(line)
        im = cv2.imread(frame_path(d, ev))
        panel = np.hstack([im, cv2.cvtColor(G[k], cv2.COLOR_GRAY2BGR), cv2.cvtColor(gt, cv2.COLOR_GRAY2BGR)])
        cv2.imwrite(os.path.join(a.out_dir, 'check', f'{seq}_eval.png'), cv2.resize(panel, None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST))

    with open(os.path.join(a.out_dir, 'guidance_summary.txt'), 'a') as fh:
        fh.write(f'# model={a.name} frames={a.frames} config={a.config}\n' + '\n'.join(summary) + '\n')
    el = time.time() - t_all
    print(f'\nDONE: {n_all} frames in {el / 60:.1f} min ({n_all / max(el, 1e-6):.1f} frames/s). Caches in {os.path.abspath(a.out_dir)}')


if __name__ == '__main__':
    main()


## 4. Locate the dataset folders (edit by hand if a path prints None)

In [ ]:
import os
def find_root(marker_dirs):
    for d, dirs, files in os.walk('/kaggle/input'):
        if all(m in dirs for m in marker_dirs):
            return d
    return None
CDNET = find_root(['baseline', 'badWeather'])
print('CDNET =', CDNET)
open('/kaggle/working/cdnet_root.txt', 'w').write(CDNET or '')

## 5. LIMU (3 sequences, about 11,600 frames). Short: run it first

In [ ]:
%%bash
cd /kaggle/working
PY=/tmp/mc/envs/mm/bin/python
$PY make_guidance_limu.py --limu_root /kaggle/input --config $(ls /tmp/psp/*.py | head -1) --ckpt $(ls /tmp/psp/*.pth | head -1) --out_dir /kaggle/working/limu_guidance_pspnet

## 6. WallFlower (7 sequences, Train+Run+Evaluate window)

In [ ]:
%%bash
cd /kaggle/working
PY=/tmp/mc/envs/mm/bin/python
$PY make_guidance_wallflower.py --wf_root /kaggle/input --config $(ls /tmp/psp/*.py | head -1) --ckpt $(ls /tmp/psp/*.pth | head -1) --name pspnet_r50_ade20k --out_dir /kaggle/working/wf_guidance_pspnet --frames window --batch 8
cat /kaggle/working/wf_guidance_pspnet/guidance_summary.txt

## 7. CDnet2014 (49 videos without PTZ, about 56,000 frames: the long run, roughly 2 h on a T4). Resumable: finished videos are skipped

In [ ]:
%%bash
cd /kaggle/working
PY=/tmp/mc/envs/mm/bin/python
$PY make_guidance_cdnet_mmseg.py --dataset_root "$(cat /kaggle/working/cdnet_root.txt)" --config $(ls /tmp/psp/*.py | head -1) --ckpt $(ls /tmp/psp/*.pth | head -1) --out_dir /kaggle/working/cdnet_guidance_pspnet --categories badWeather,baseline,cameraJitter,dynamicBackground,intermittentObjectMotion,lowFramerate,nightVideos,shadow,thermal,turbulence --stride 2 --max_dim 320

## 8. Quick check: frames and foreground share per cache (a cache with ~0% foreground on a people/car video is broken)

In [ ]:
import numpy as np, glob, os
for p in sorted(glob.glob('/kaggle/working/*pspnet*/*.npz')):
    z = np.load(p); g = z['guid']
    print(f"{os.path.relpath(p, '/kaggle/working'):60s} N={len(z['nums']):5d}  {g.shape[1]}x{g.shape[2]}  fg {100*(g[::max(1,len(g)//50)]>127).mean():5.1f}%  prob max {float(z['prob'][::max(1,len(g)//50)].max()):.2f}")

## 9. Zip for download (CDnet zipped per category and moved, to stay under the 20 GB output limit)

In [ ]:
%%bash
cd /kaggle/working
zip -q -r limu_guidance_pspnet.zip limu_guidance_pspnet && zip -q -r wf_guidance_pspnet.zip wf_guidance_pspnet
for c in $(echo badWeather,baseline,cameraJitter,dynamicBackground,intermittentObjectMotion,lowFramerate,nightVideos,shadow,thermal,turbulence | tr ',' ' '); do ls cdnet_guidance_pspnet/${c}__*.npz >/dev/null 2>&1 && zip -q -m -j cdnet_pspnet_${c}.zip cdnet_guidance_pspnet/${c}__*.npz; done
ls -lh *.zip